# Gikomba Market Stall Performance Analysis

**Analyst:** Collins Kipngetich Basuben
**Client:** Wanjiku, fresh-produce stall at Gikomba Market

Wanjiku wrote down two weeks of daily sales and a rough stock record. She wants to know how the business is doing, whether it is growing, what to restock and where to focus. This notebook answers those questions using core Python only (no pandas or NumPy).

## Dataset

The data below is copied exactly as provided. `daily_sales` has 14 days of sales in KES, `days` gives the day name for each position, and `stock` holds quantity, reorder level, cost and price for each product.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Daily sales summary

I use `sum()`, `max()` and `min()` on the 14 days of sales, and divide the total by the number of days for the average. To find which day was best and worst, I use `list.index()` to get the position of the highest and lowest sales. That position gives the day name from the `days` list, and positions 0 to 6 are week 1 while 7 to 13 are week 2.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

# list.index() returns the position of the first match, which tells us the day and the week
best_position = daily_sales.index(highest_sales)
worst_position = daily_sales.index(lowest_sales)

best_day = days[best_position]
worst_day = days[worst_position]
best_week = best_position // 7 + 1
worst_week = worst_position // 7 + 1

print(f"Total sales:    KES {total_sales:,}")
print(f"Average sales:  KES {average_sales:,.2f}")
print(f"Highest sales:  KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest sales:   KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales:    KES 46,450
Average sales:  KES 3,317.86
Highest sales:  KES 5,600 (Sat, week 2)
Lowest sales:   KES 1,950 (Tue, week 1)


## Task 2: Week-on-week growth

The function `week_totals()` slices the first seven days and the last seven days, adds each slice up and returns both totals as a tuple. I unpack the tuple into two variables, then calculate growth as the change from week 1 to week 2 divided by the week 1 total.

In [3]:
def week_totals(sales):
    """Return the sales totals for week 1 and week 2 as a tuple.

    Parameters:
        sales (list): 14 daily sales figures in KES, starting on a Monday.

    Returns:
        tuple: (week 1 total, week 2 total)
    """
    week_1 = sales[:7]      # positions 0 to 6
    week_2 = sales[7:14]    # positions 7 to 13
    return sum(week_1), sum(week_2)


week_1_total, week_2_total = week_totals(daily_sales)
growth_pct = (week_2_total - week_1_total) / week_1_total * 100

print(f"Week 1 total:  KES {week_1_total:,}")
print(f"Week 2 total:  KES {week_2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.2f}%")

Week 1 total:  KES 22,050
Week 2 total:  KES 24,400
Week-on-week growth: +10.66%


## Task 3: Classifying each day

The function `classify_day()` compares one day's sales with the average. More than 20% above the average is "Strong", more than 20% below is "Weak", and anything else is "Normal". I apply it to every day and count the results with the dictionary counting pattern (`counts.get(label, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Label a day's sales as Strong, Normal or Weak compared with the average.

    Parameters:
        amount (int or float): sales for the day in KES.
        average (float): average daily sales in KES.

    Returns:
        str: "Strong" if amount is more than 20% above average,
             "Weak" if more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    return "Normal"


category_counts = {}
print(f"{'Day':<6}{'Week':<6}{'Sales (KES)':>12}   Category")
for position, (day, amount) in enumerate(zip(days, daily_sales)):
    label = classify_day(amount, average_sales)
    category_counts[label] = category_counts.get(label, 0) + 1
    print(f"{day:<6}{position // 7 + 1:<6}{amount:>12,}   {label}")

print()
print("Days per category:", category_counts)

Day   Week   Sales (KES)   Category
Mon   1            2,300   Weak
Tue   1            1,950   Weak
Wed   1            2,800   Normal
Thu   1            2,100   Weak
Fri   1            3,600   Normal
Sat   1            5,200   Strong
Sun   1            4,100   Strong
Mon   2            2,500   Weak
Tue   2            2,200   Weak
Wed   2            3,100   Normal
Thu   2            2,400   Weak
Fri   2            3,900   Normal
Sat   2            5,600   Strong
Sun   2            4,700   Strong

Days per category: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

I loop through the days and sales together. Saturday and Sunday sales go into one list and Monday to Friday sales into another. Then I compare the two averages and express the gap as a percentage of the weekday average.

In [5]:
weekend_days = ("Sat", "Sun")
weekday_sales = []
weekend_sales = []

for day, amount in zip(days, daily_sales):
    if day in weekend_days:
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_average = sum(weekday_sales) / len(weekday_sales)
weekend_average = sum(weekend_sales) / len(weekend_sales)
weekend_gap_pct = (weekend_average - weekday_average) / weekday_average * 100

print(f"Weekday average (Mon-Fri): KES {weekday_average:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_average:,.2f}")
print(f"Weekends sell {weekend_gap_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends sell 82.5% more per day than weekdays


## Task 5: Stock check and gross margin

The function `items_to_reorder()` loops through the stock dictionary with `.items()` and collects every product whose quantity is at or below its reorder level. A second small function calculates gross margin as (price - cost) divided by price, and I use it for each product.

In [6]:
def items_to_reorder(stock):
    """Return the products that need restocking.

    Parameters:
        stock (dict): product name -> dict with "qty" and "reorder" in kg.

    Returns:
        list: names of products whose quantity is at or below the reorder level.
    """
    low_stock = []
    for product, details in stock.items():
        if details["qty"] <= details["reorder"]:
            low_stock.append(product)
    return low_stock


def gross_margin_pct(price, cost):
    """Return gross margin as a percentage: (price - cost) / price * 100.

    Parameters:
        price (int or float): selling price in KES per kg.
        cost (int or float): buying cost in KES per kg.

    Returns:
        float: gross margin percentage.
    """
    return (price - cost) / price * 100


reorder_list = items_to_reorder(stock)

margins = {}
for product, details in stock.items():
    margins[product] = gross_margin_pct(details["price"], details["cost"])

print("Products to reorder:", reorder_list)
print()
for product, margin in margins.items():
    print(f"{product:<10} gross margin: {margin:.1f}%")

Products to reorder: ['Tomatoes', 'Sukuma']

Tomatoes   gross margin: 33.3%
Onions     gross margin: 30.0%
Sukuma     gross margin: 40.0%
Potatoes   gross margin: 33.3%
Avocado    gross margin: 42.9%


## Task 6: Stall Performance Report

I bring all the results together in one printed report. Money is written as KES with thousands separators (`:,`), and the columns are aligned with fixed widths so the report is easy to read. A small helper function `kes()` keeps the money formatting in one place.

In [7]:
def kes(amount, decimals=0):
    """Format an amount as KES with thousands separators.

    Parameters:
        amount (int or float): the amount in KES.
        decimals (int): number of decimal places to show.

    Returns:
        str: for example "KES 46,450" or "KES 3,317.86".
    """
    return f"KES {amount:,.{decimals}f}"


WIDTH = 60
LINE = "=" * WIDTH
THIN = "-" * WIDTH

print(LINE)
print(f"{'STALL PERFORMANCE REPORT':^{WIDTH}}")
print(f"{'Wanjiku, Fresh Produce, Gikomba Market':^{WIDTH}}")
print(LINE)

print("\nSALES SUMMARY (14 days)")
print(THIN)
print(f"{'Total sales':<30}{kes(total_sales):>30}")
print(f"{'Average daily sales':<30}{kes(average_sales, 2):>30}")
print(f"{'Best day':<30}{f'{best_day}, week {best_week}: ' + kes(highest_sales):>30}")
print(f"{'Weakest day':<30}{f'{worst_day}, week {worst_week}: ' + kes(lowest_sales):>30}")

print("\nWEEKLY GROWTH")
print(THIN)
print(f"{'Week 1 total':<30}{kes(week_1_total):>30}")
print(f"{'Week 2 total':<30}{kes(week_2_total):>30}")
print(f"{'Week-on-week growth':<30}{f'{growth_pct:+.2f}%':>30}")

print("\nDAY CLASSIFICATION")
print(THIN)
for label in ("Strong", "Normal", "Weak"):
    print(f"{label + ' days':<30}{category_counts.get(label, 0):>30}")

print("\nWEEKDAYS VS WEEKENDS")
print(THIN)
print(f"{'Weekday average (Mon-Fri)':<30}{kes(weekday_average, 2):>30}")
print(f"{'Weekend average (Sat-Sun)':<30}{kes(weekend_average, 2):>30}")
print(f"{'Weekend premium':<30}{f'{weekend_gap_pct:.1f}%':>30}")

print("\nSTOCK STATUS")
print(THIN)
print(f"{'Product':<12}{'In stock':>10}{'Reorder':>10}{'Margin':>12}{'Action':>16}")
for product, details in stock.items():
    action = "REORDER" if product in reorder_list else "OK"
    print(f"{product:<12}{details['qty']:>7} kg{details['reorder']:>7} kg"
          f"{margins[product]:>11.1f}%{action:>16}")

print("\nRestock now:", ", ".join(reorder_list))
print(LINE)

                  STALL PERFORMANCE REPORT                  
           Wanjiku, Fresh Produce, Gikomba Market           

SALES SUMMARY (14 days)
------------------------------------------------------------
Total sales                                       KES 46,450
Average daily sales                             KES 3,317.86
Best day                              Sat, week 2: KES 5,600
Weakest day                           Tue, week 1: KES 1,950

WEEKLY GROWTH
------------------------------------------------------------
Week 1 total                                      KES 22,050
Week 2 total                                      KES 24,400
Week-on-week growth                                  +10.66%

DAY CLASSIFICATION
------------------------------------------------------------
Strong days                                                4
Normal days                                                4
Weak days                                                  6

WEEKDAYS VS WEEKENDS
---

## Task 7: Business recommendations

I look at what the numbers say about growth, the best days to focus on, stock that is running low and the products that earn the most per kilogram. Each recommendation below is tied to a figure from the analysis above.

## Recommendations for Wanjiku

Your business is growing: week 2 sales were KES 24,400 compared with KES 22,050 in week 1, which is 10.66% higher, so keep writing down your daily sales to watch this trend. Focus your effort on weekends, because Saturday and Sunday average KES 4,900 a day against KES 2,685 on weekdays (82.5% more) and all four of your "Strong" days fell on a weekend. Restock Tomatoes (18 kg against a reorder level of 25 kg) and Sukuma (9 kg against 15 kg) before Friday so you do not run out on your best selling days. Give Avocado the best spot on your stall, since it has the highest margin at 42.9% (KES 60 profit on every kg) and you have only 12 kg, just above its reorder level of 10 kg. Six of the 14 days were "Weak", all of them Monday, Tuesday or Thursday, so use those slow days to move your extra Potatoes (55 kg in stock against a reorder level of 30 kg) with a small discount.